# Accessibility Data Preparation (Refactored)

**Changes from 3-accessibility-data-prep.ipynb:**
- Creates ONE origins file with ALL workers (no pre-grouping by commute time)
- Supports sensitivity analysis with different time budgets (60, 75, 90, 120 min)
- Filtering by time budget done in post-processing (notebook 7b)

In [1]:
%load_ext autoreload
%autoreload 2
%cd D:\netmob25

D:\netmob25


In [2]:
import os
os.environ['USE_PYGEOS'] = '0'
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

## Configuration

In [3]:
# Output directory for refactored pipeline
OUTPUT_DIR = Path("dbs/sp_accessibility_v2/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Max trip duration for r5r (covers all sensitivity scenarios)
MAX_TRIP_DURATION = 120  # minutes

print(f"Output directory: {OUTPUT_DIR}")
print(f"Max trip duration: {MAX_TRIP_DURATION} min")

Output directory: dbs\sp_accessibility_v2\data
Max trip duration: 120 min


## 1. Load boundary data

In [4]:
# Load Paris region boundary
gdf_boundary = gpd.read_file('dbs/geo/paris_iris.geojson')
gdf_boundary = gdf_boundary.to_crs(epsg=4326)
boundary = gdf_boundary.union_all().buffer(0.01)  # Small buffer
print(f"Boundary loaded: {len(gdf_boundary)} IRIS zones")

Boundary loaded: 5264 IRIS zones


## 2. Prepare work locations as origins (ALL workers)

In [5]:
# Load commuter trips to get work locations
df_trips = pd.read_csv('dbs/data_p/commuter_trips.csv')
print(f"Total trips: {len(df_trips)}")
print(f"Unique individuals: {df_trips['ID'].nunique()}")

Total trips: 52965
Unique individuals: 2457


In [6]:
# Extract work locations (destination of WORK trips)
df_work = df_trips[df_trips['purpose_d'] == 'WORK'].copy()
df_work = df_work.drop_duplicates(subset=['ID'])
df_work = df_work[['ID', 'end_lon', 'end_lat']].rename(
    columns={'ID': 'id', 'end_lon': 'lon', 'end_lat': 'lat'}
)
print(f"Unique work locations: {len(df_work)}")

Unique work locations: 2442


In [7]:
# Save ALL work locations as origins (single file)
origins_file = OUTPUT_DIR / "origins_work.csv"
df_work[['id', 'lon', 'lat']].to_csv(origins_file, index=False)
print(f"Saved {len(df_work)} origins to {origins_file}")

Saved 2442 origins to dbs\sp_accessibility_v2\data\origins_work.csv


## 3. Prepare home locations as destinations

In [8]:
# Extract home locations (destination of HOME trips)
df_home = df_trips[df_trips['purpose_d'] == 'HOME'].copy()
df_home = df_home.drop_duplicates(subset=['ID'])
df_home = df_home[['ID', 'end_lon', 'end_lat']].rename(
    columns={'ID': 'id', 'end_lon': 'lon', 'end_lat': 'lat'}
)
print(f"Unique home locations: {len(df_home)}")

Unique home locations: 2453


In [9]:
# Save home locations
homes_file = OUTPUT_DIR / "destinations_home.csv"
df_home[['id', 'lon', 'lat']].to_csv(homes_file, index=False)
print(f"Saved {len(df_home)} home locations to {homes_file}")

Saved 2453 home locations to dbs\sp_accessibility_v2\data\destinations_home.csv


## 4. Prepare LEISURE POIs as destinations

In [10]:
# Load POIs
gdf_poi = gpd.read_file('dbs/geo/pois_p.gpkg', layer='pois')
gdf_poi = gdf_poi.to_crs(epsg=4326)
print(f"Total POIs: {len(gdf_poi)}")
print(f"POI purposes: {gdf_poi['purpose'].unique()}")

Total POIs: 2098361
POI purposes: ['OTHER' 'LEISURE' 'PURCHASE' 'HEALTH']


In [11]:
# Filter to LEISURE only
gdf_leisure = gdf_poi[gdf_poi['purpose'] == 'LEISURE'].copy()
print(f"LEISURE POIs: {len(gdf_leisure)}")

# Keep only POIs within boundary
gdf_leisure = gdf_leisure[gdf_leisure.within(boundary)]
print(f"LEISURE POIs within boundary: {len(gdf_leisure)}")

LEISURE POIs: 567513
LEISURE POIs within boundary: 43576


In [12]:
# Add coordinates and save
gdf_leisure['lon'] = gdf_leisure.geometry.x
gdf_leisure['lat'] = gdf_leisure.geometry.y

pois_file = OUTPUT_DIR / "destinations_leisure.csv"
gdf_leisure[['id', 'lon', 'lat']].to_csv(pois_file, index=False)
print(f"Saved {len(gdf_leisure)} LEISURE POIs to {pois_file}")

Saved 43576 LEISURE POIs to dbs\sp_accessibility_v2\data\destinations_leisure.csv


## 5. Load and save time budget data

In [13]:
# Load commuter time budget (for filtering in notebook 7b)
df_budget = pd.read_csv('dbs/data_p/commuter_time_budget.csv')
print(f"Time budget data: {len(df_budget)} individuals")
print(df_budget[['time_budget', 'time_hw', 'tt_wkh']].describe())

Time budget data: 2457 individuals
       time_budget      time_hw       tt_wkh
count  2457.000000  2294.000000  2294.000000
mean     94.794668    40.515911     8.968178
std      49.692735    25.134394    50.268788
min       5.000000     1.500000  -628.000000
25%      60.000000    22.000000   -20.000000
50%      86.000000    36.000000    18.000000
75%     120.000000    55.000000    46.000000
max     713.500000   359.000000    87.000000


In [14]:
# Copy to output directory for easy access
budget_file = OUTPUT_DIR / "time_budget.csv"
df_budget.to_csv(budget_file, index=False)
print(f"Saved time budget to {budget_file}")

Saved time budget to dbs\sp_accessibility_v2\data\time_budget.csv


## Summary

In [15]:
print("=" * 50)
print("Data preparation complete!")
print("=" * 50)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nFiles created:")
print(f"  - origins_work.csv: {len(df_work)} work locations")
print(f"  - destinations_home.csv: {len(df_home)} home locations")
print(f"  - destinations_leisure.csv: {len(gdf_leisure)} LEISURE POIs")
print(f"  - time_budget.csv: {len(df_budget)} individuals")
print(f"\nNext step: Run 4b-sp-accessibility-wk.R")

Data preparation complete!

Output directory: dbs\sp_accessibility_v2\data

Files created:
  - origins_work.csv: 2442 work locations
  - destinations_home.csv: 2453 home locations
  - destinations_leisure.csv: 43576 LEISURE POIs
  - time_budget.csv: 2457 individuals

Next step: Run 4b-sp-accessibility-wk.R
